# 03 – State Design

Visualise per-appliance power histograms and the discretised ON/OFF (+ multi-level) states used for HMM training.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from src.load_refit import load_config
from src.states import label_states

cfg = load_config('../configs/config.yaml')
PROCESSED_DIR = pathlib.Path('../') / cfg['data']['processed_dir']
house_id = cfg['houses'][0]

df = pd.read_csv(PROCESSED_DIR / f'house_{house_id}.csv', index_col='timestamp', parse_dates=True)
df.head()

In [ ]:
appliances = cfg['appliances']

fig, axes = plt.subplots(len(appliances), 2, figsize=(14, 3*len(appliances)))
for i, (app, app_cfg) in enumerate(appliances.items()):
    if app not in df.columns:
        continue
    series = df[app].dropna()
    labels, centroids = label_states(series, app_cfg['on_threshold'], app_cfg['n_states'])

    # Histogram
    axes[i, 0].hist(series, bins=100, log=True, color='steelblue', edgecolor='none')
    axes[i, 0].axvline(app_cfg['on_threshold'], color='red', ls='--', label='ON threshold')
    axes[i, 0].set_title(f'{app} – power distribution')
    axes[i, 0].legend()

    # State sequence (first 1 day = 1440 min)
    axes[i, 1].plot(labels[:1440], lw=0.6, color='darkorange')
    axes[i, 1].set_title(f'{app} – state labels (1 day)')
    axes[i, 1].set_yticks(range(app_cfg['n_states']))

plt.tight_layout()
plt.show()